In [ ]:
import sys
import subprocess

required = ["transformers", "datasets", "scipy", "pandas", "torch", "accelerate"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/stsb-distilroberta-base"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
subset_size = 512
max_length = 128
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "subset_size": subset_size,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
ds = ds.select(range(min(subset_size, len(ds))))
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print({
    "num_labels": int(getattr(model.config, "num_labels", -1)),
    "problem_type": getattr(model.config, "problem_type", None),
    "id2label": getattr(model.config, "id2label", None),
    "device": str(next(model.parameters()).device),
})


In [ ]:
sentence1_list = df["sentence1"].tolist()
sentence2_list = df["sentence2"].tolist()

all_logits = []

with torch.no_grad():
    for start in range(0, len(df), batch_size):
        end = min(start + batch_size, len(df))
        batch_s1 = sentence1_list[start:end]
        batch_s2 = sentence2_list[start:end]
        inputs = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        logits = outputs.logits.detach().float().cpu().numpy()
        all_logits.append(logits)

raw_logits = np.concatenate(all_logits, axis=0)
if raw_logits.ndim == 2 and raw_logits.shape[1] == 1:
    raw_scores = raw_logits[:, 0].astype(np.float32)
else:
    raw_scores = raw_logits.squeeze().astype(np.float32)

predicted_score_0_5 = np.clip(raw_scores, 0.0, 5.0).astype(np.float32)

print(pd.DataFrame({
    "raw_logit": raw_scores[:10],
    "predicted_score_0_5": predicted_score_0_5[:10],
} ))


In [ ]:
labels = df["label"].to_numpy(dtype=np.float32)

pearson_value = pearsonr(predicted_score_0_5, labels).statistic
spearman_value = spearmanr(predicted_score_0_5, labels).statistic
mae_value = np.mean(np.abs(predicted_score_0_5 - labels))

results_df = df.copy()
results_df["raw_logit"] = raw_scores
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["abs_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])

compact_cols = ["sentence1", "sentence2", "label", "raw_logit", "predicted_score_0_5", "abs_error"]

best_pairs = results_df.sort_values(["abs_error", "label"], ascending=[True, False]).head(5)[compact_cols]
worst_pairs = results_df.sort_values(["abs_error", "label"], ascending=[False, False]).head(5)[compact_cols]

print("best_pairs")
print(best_pairs.to_string(index=False))
print()
print("worst_pairs")
print(worst_pairs.to_string(index=False))


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"pearson_prediction: {pearson_value:.6f}")
print(f"spearman_prediction: {spearman_value:.6f}")
print(f"mae_prediction: {mae_value:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
